### **均線法則**

In [ ]:
# @title
import datetime
import time
from google.colab import userdata
import pandas as pd
import pytz
import requests
import yfinance as yf

# ==========================================
# 1. 憑證設定 (優先讀取 Colab Secrets)
# ==========================================
try:
  LINE_CHANNEL_ACCESS_TOKEN = userdata.get('LINE_TOKEN')
  LINE_USER_ID = userdata.get('LINE_USER_ID')
except Exception:
  # 若未於 Secrets 設定 LINE_TOKEN 與 LINE_USER_ID，可在此貼上字串
  LINE_CHANNEL_ACCESS_TOKEN = "/JvwwTYrzi4Ao4twBsWBuo1jTNCpAHIWEVxLBVXfjduSdsKMbvnt+HBBZN2q/vuMobm4O2M0U1P8s8rdl8t8bBp6sl9bO4KMTY09VOas14v4O+ao8VNPuqquWekeL755BFG7fQ1y/bYlbnOH/TNbngdB04t89/1O/w1cDnyilFU="
  LINE_USER_ID = "U13239c411c6e3bd5c44ff791f8f35e37"

TW_TZ = pytz.timezone("Asia/Taipei")

# ==========================================
# 2. 39 檔監控股票清單與均線參數
# ==========================================
STOCKS = {
    "1504": {"name": "東元", "ma": 23, "type": ".TW"},
    "1711": {"name": "永光", "ma": 30, "type": ".TW"},
    "1795": {"name": "美時", "ma": 23, "type": ".TW"},
    "2301": {"name": "光寶科", "ma": 20, "type": ".TW"},
    "2303": {"name": "聯電", "ma": 27, "type": ".TW"},
    "2308": {"name": "台達電", "ma": 26, "type": ".TW"},
    "2313": {"name": "華通", "ma": 19, "type": ".TW"},
    "2317": {"name": "鴻海", "ma": 19, "type": ".TW"},
    "2324": {"name": "仁寶", "ma": 19, "type": ".TW"},
    "2327": {"name": "國巨", "ma": 19, "type": ".TW"},
    "2330": {"name": "台積電", "ma": 26, "type": ".TW"},
    "2337": {"name": "旺宏", "ma": 30, "type": ".TW"},
    "2338": {"name": "光罩", "ma": 26, "type": ".TW"},
    "2344": {"name": "華邦電", "ma": 19, "type": ".TW"},
    "2345": {"name": "智邦", "ma": 30, "type": ".TW"},
    "2347": {"name": "聯強", "ma": 38, "type": ".TW"},
    "2353": {"name": "宏碁", "ma": 17, "type": ".TW"},
    "2367": {"name": "耀華", "ma": 26, "type": ".TW"},
    "2376": {"name": "技嘉", "ma": 30, "type": ".TW"},
    "2382": {"name": "廣達", "ma": 23, "type": ".TW"},
    "2383": {"name": "台光電", "ma": 22, "type": ".TW"},
    "2385": {"name": "群光", "ma": 25, "type": ".TW"},
    "2395": {"name": "研華", "ma": 22, "type": ".TW"},
    "2408": {"name": "南亞科", "ma": 14, "type": ".TW"},
    "2409": {"name": "友達", "ma": 18, "type": ".TW"},
    "2454": {"name": "聯發科", "ma": 29, "type": ".TW"},
    "2472": {"name": "立隆電", "ma": 24, "type": ".TW"},
    "2485": {"name": "兆赫", "ma": 27, "type": ".TW"},
    "2498": {"name": "宏達電", "ma": 23, "type": ".TW"},
    "2610": {"name": "華航", "ma": 17, "type": ".TW"},
    "2646": {"name": "星宇航空", "ma": 23, "type": ".TW"},
    "3008": {"name": "大立光", "ma": 25, "type": ".TW"},
    "3017": {"name": "奇鋐", "ma": 19, "type": ".TW"},
    "3231": {"name": "緯創", "ma": 26, "type": ".TW"},
    "3450": {"name": "聯鈞", "ma": 20, "type": ".TW"},
    "3481": {"name": "群創", "ma": 20, "type": ".TW"},
    "3707": {"name": "漢磊", "ma": 18, "type": ".TW"},
    "3711": {"name": "日月光投控", "ma": 31, "type": ".TW"},
    "5347": {"name": "世界", "ma": 28, "type": ".TW"},
    "6116": {"name": "彩晶", "ma": 16, "type": ".TW"},
    "6191": {"name": "精成科", "ma": 18, "type": ".TW"},
    "6284": {"name": "佳邦", "ma": 24, "type": ".TW"},
    "6443": {"name": "元晶", "ma": 19, "type": ".TW"},
    "8069": {"name": "元太", "ma": 22, "type": ".TWO"},
}


# ==========================================
# 3. LINE 推播函式
# ==========================================
def send_line_push(msg_text):
  url = "https://api.line.me/v2/bot/message/push"
  headers = {
      "Content-Type": "application/json",
      "Authorization": f"Bearer {LINE_CHANNEL_ACCESS_TOKEN}",
  }
  payload = {
      "to": LINE_USER_ID,
      "messages": [{"type": "text", "text": msg_text}],
  }
  try:
    res = requests.post(url, headers=headers, json=payload, timeout=10)
    return res.status_code == 200
  except Exception as e:
    print(f"LINE 發送失敗: {e}")
    return False


# ==========================================
# 4. 盤中數據計算與條件觸發檢查
# ==========================================
def check_stock_alerts(alerted_today):
  now_tw = datetime.datetime.now(TW_TZ)
  print(
      f"[{now_tw.strftime('%Y-%m-%d %H:%M:%S')}] 開始掃描 39 檔台股價格..."
  )

  for code, info in STOCKS.items():
    if code in alerted_today:
      continue  # 當日已觸發提醒則跳過，防止重複洗版

    ticker_symbol = f"{code}{info['type']}"
    ma_period = info["ma"]
    stock_name = info["name"]

    try:
      stock = yf.Ticker(ticker_symbol)
      df = stock.history(period="60d", interval="1d")

      if len(df) < ma_period + 1:
        continue

      # 前一日均值 (計算不包含當日盤中即時日 K 的歷史數據)
      yesterday_df = df.iloc[:-1]
      prev_ma = yesterday_df["Close"].tail(ma_period).mean()
      threshold_price = round(prev_ma * 1.01, 2)  # 突破目標價

      current_price = round(df["Close"].iloc[-1], 2)
      current_vol = int(df["Volume"].iloc[-1] // 1000)
      yesterday_vol = int(df["Volume"].iloc[-2] // 1000)

      # 觸發條件：當前股價 >= 前一日均值 1.01 倍
      if current_price >= threshold_price:
        message = (
            f"📈 【台股均線突破提醒】\n"
            f"------------------------\n"
            f"1. 股票名稱 (代號)：{stock_name} ({code})\n"
            f"2. 當前股價：{current_price} 元\n"
            f"3. 突破價格 (前日{ma_period}MA x 1.01)：{threshold_price} 元\n"
            f"4. 成交量資訊：\n"
            f"   • 當前成交量：{current_vol:,} 張\n"
            f"   • 昨天成交量：{yesterday_vol:,} 張"
        )

        if send_line_push(message):
          print(f"✅ 成功發送推播：{stock_name} ({code})")
          alerted_today.add(code)

    except Exception as e:
      print(f"❌ 處理 {code} {stock_name} 時出錯: {e}")

    time.sleep(0.3)


# ==========================================
# 5. 主執行迴圈 (開盤時間監控)
# ==========================================
alerted_today = set()
last_reset_day = datetime.datetime.now(TW_TZ).date()

print("🚀 盤中均線監控系統已啟動...")

while True:
  now_tw = datetime.datetime.now(TW_TZ)

  # 每日跨日重置提醒清單
  if now_tw.date() != last_reset_day:
    alerted_today.clear()
    last_reset_day = now_tw.date()

  # 台股交易時間：週一至週五 09:00 - 13:30
  is_weekday = now_tw.weekday() < 5
  in_trading_hours = (
      (now_tw.hour == 9 and now_tw.minute >= 0)
      or (10 <= now_tw.hour <= 12)
      or (now_tw.hour == 13 and now_tw.minute <= 30)
  )

  if is_weekday and in_trading_hours:
    check_stock_alerts(alerted_today)
    time.sleep(300)  # 盤中每 5 分鐘掃描一次
  else:
    print(
        f"[{now_tw.strftime('%H:%M:%S')}] 非交易時間，等待中... (每 10"
        " 分鐘檢查一次)"
    )
    time.sleep(600)

🚀 盤中均線監控系統已啟動...
[22:56:21] 非交易時間，等待中... (每 10 分鐘檢查一次)


KeyboardInterrupt: 